# FAERS EDA: Complete Analysis of ALL Quarters (2019-2024)

Comprehensive analysis of FDA Adverse Event Reporting System data with df.head() displays for all file types.

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
plt.style.use('bmh')
faers_path = r'D:\ADE DATASET DOWNLOAD\FAERS\FAERSdata'
eda_path = r'D:\ADE DATASET DOWNLOAD\EDA'

quarters = [f"{y%100:02d}Q{q}" for y in range(2019, 2025) for q in range(1, 5)]

## 1. Data Loading Function

In [ ]:
def load_faers_batch(prefix, quarters_subset, sample_n=None):
    all_dfs = []
    for q in quarters_subset:
        filename = f"{prefix}{q}.txt"
        path = os.path.join(faers_path, filename)
        if os.path.exists(path):
            try:
                df = pd.read_csv(path, sep='$', low_memory=False)
                if sample_n:
                    df = df.sample(min(len(df), sample_n), random_state=42)
                df['quarter'] = q
                all_dfs.append(df)
            except Exception as e:
                print(f"Error loading {filename}: {e}")
    
    return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

## 2. DEMO Files - Demographics & Reporter Data

In [ ]:
demo = load_faers_batch('DEMO', quarters[:8], sample_n=10000)
print(f"Total DEMO Records (sampled): {len(demo):,}")
display(demo.head(5))

if not demo.empty and 'occp_cod' in demo.columns:
    reporter_map = {'MD': 'Physician', 'PH': 'Pharmacist', 'OT': 'Other Healthcare', 
                    'HP': 'Healthcare Prof', 'CN': 'Consumer', 'LW': 'Lawyer'}
    demo['reporter_type'] = demo['occp_cod'].map(reporter_map).fillna('Other/Unknown')
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.countplot(data=demo, y='reporter_type', ax=axes[0], palette='viridis')
    axes[0].set_title('Reporter Type Distribution')
    
    if 'age' in demo.columns:
        sns.histplot(demo['age'].dropna(), bins=30, kde=True, ax=axes[1])
        axes[1].set_title('Age Distribution')
    
    plt.tight_layout()
    plt.show()

## 3. DRUG Files - Drug Information

In [ ]:
drug = load_faers_batch('DRUG', quarters[:4], sample_n=15000)
print(f"Total DRUG Records (sampled): {len(drug):,}")
display(drug.head(5))

if not drug.empty and 'drugname' in drug.columns:
    top_drugs = drug['drugname'].value_counts().head(20)
    plt.figure(figsize=(12, 8))
    sns.barplot(x=top_drugs.values, y=top_drugs.index, palette='rocket')
    plt.title('Top 20 Drugs in FAERS Reports')
    plt.tight_layout()
    plt.show()

## 4. REAC Files - Adverse Reactions

In [ ]:
reac = load_faers_batch('REAC', quarters[:4], sample_n=15000)
print(f"Total REAC Records (sampled): {len(reac):,}")
display(reac.head(5))

if not reac.empty and 'pt' in reac.columns:
    top_reactions = reac['pt'].value_counts().head(20)
    plt.figure(figsize=(12, 8))
    sns.barplot(x=top_reactions.values, y=top_reactions.index, palette='magma')
    plt.title('Top 20 Adverse Reactions')
    plt.tight_layout()
    plt.show()

## 5. Drug-Reaction Signal Detection

In [ ]:
if not drug.empty and not reac.empty:
    merged = pd.merge(drug[['primaryid', 'drugname']], 
                      reac[['primaryid', 'pt']], 
                      on='primaryid')
    
    top_drugs_sig = merged['drugname'].value_counts().head(15).index
    top_reacs_sig = merged['pt'].value_counts().head(15).index
    
    pivot_table = merged[merged['drugname'].isin(top_drugs_sig) & merged['pt'].isin(top_reacs_sig)].pivot_table(
        index='drugname', columns='pt', values='primaryid', aggfunc='count', fill_value=0
    )

    plt.figure(figsize=(16, 10))
    sns.heatmap(pivot_table, annot=True, fmt='d', cmap='YlOrRd')
    plt.title('Drug-Reaction Co-occurrence Heatmap')
    plt.tight_layout()
    plt.show()

## 6. OUTC Files - Outcomes

In [ ]:
outc = load_faers_batch('OUTC', ['19Q1', '21Q2', '21Q3'], sample_n=30000)
if not outc.empty:
    print(f"Total OUTC Records: {len(outc):,}")
    display(outc.head(5))
    
    outcome_map = {'DE': 'Death', 'LT': 'Life-Threatening', 'HO': 'Hospitalization',
                   'DS': 'Disability', 'CA': 'Congenital Anomaly', 'RI': 'Required Intervention'}
    if 'outc_cod' in outc.columns:
        outc['outcome'] = outc['outc_cod'].map(outcome_map).fillna('Other')
        
        plt.figure(figsize=(10, 6))
        sns.countplot(data=outc, x='outcome', palette='Set2', order=outc['outcome'].value_counts().index)
        plt.title('Patient Outcomes Distribution')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 7. INDI Files - Indications (if available)

In [ ]:
indi_path = os.path.join(faers_path, 'INDI21Q2.txt')
if os.path.exists(indi_path):
    indi = pd.read_csv(indi_path, sep='$', nrows=10000)
    print(f"Total INDI Records (sample): {len(indi):,}")
    display(indi.head(5))
else:
    print("INDI files not found (may only be available in certain quarters)")

## 8. THER Files - Therapy (if available)

In [ ]:
ther_path = os.path.join(faers_path, 'THER21Q2.txt')
if os.path.exists(ther_path):
    ther = pd.read_csv(ther_path, sep='$', nrows=10000)
    print(f"Total THER Records (sample): {len(ther):,}")
    display(ther.head(5))
else:
    print("THER files not found (may only be available in certain quarters)")

## 9. Temporal Trends Analysis

In [ ]:
if not demo.empty and 'quarter' in demo.columns:
    temporal_trends = demo.groupby('quarter').size().reset_index(name='report_count')
    
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=temporal_trends, x='quarter', y='report_count', marker='o')
    plt.title('FAERS Reporting Trends Over Time')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 10. Summary Report

In [ ]:
print("\n" + "="*60)
print("FAERS ANALYSIS SUMMARY")
print("="*60)
print(f"DEMO records analyzed: {len(demo):,}")
print(f"DRUG records analyzed: {len(drug):,}")
print(f"REAC records analyzed: {len(reac):,}")
print(f"OUTC records analyzed: {len(outc):,}")
print(f"Quarters covered: {len(quarters[:8])}")
print("="*60)